# 01 · Continued pre-training (domain adaptation) on health text
Runtime → **T4 GPU**. Trains a LoRA adapter on the base model with the ordinary next-token objective over PubMed Central articles, MedlinePlus and the Ghana STG/EML. The validation loss curve is your evidence that the model is learning. Checkpoints go to Drive; re-running resumes.

In [ ]:
!pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install -q -U "transformers>=5"      # Qwen3.5 needs transformers v5; restart the runtime if Colab asks, then skip this cell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, zipfile
BASE = '/content/drive/MyDrive/health-llm'            # put colab_health_bundle.zip here
os.makedirs(BASE, exist_ok=True)
if not os.path.exists('cpt_train.jsonl'):
    zipfile.ZipFile(f'{BASE}/colab_health_bundle.zip').extractall('.')
sys.path.insert(0, '.')
MODEL = 'unsloth/Qwen3.5-2B-Base'    # base (pre-trained only) model, the right start for continued pre-training. 4B does not fit a T4: Unsloth forces float32 on Qwen3.5 there (no bf16), ~16 GB of weights vs 15 GB VRAM
LOAD = dict(load_in_4bit=False, load_in_16bit=True, full_finetuning=False)   # Unsloth advises against 4-bit on Qwen3.5
LORA = dict(r=32, lora_alpha=64, lora_dropout=0, use_gradient_checkpointing='unsloth', random_state=3407,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
!nvidia-smi -L

In [ ]:
import glob, json, torch, hl_lib
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import Trainer, TrainingArguments, default_data_collator
from datasets import Dataset

BLOCK, TOKEN_BUDGET = 2048, 6_000_000      # T4: budget decides run time (fp32 on a T4 is slow). Lower it to finish sooner.
model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=BLOCK, **LOAD)
model = FastLanguageModel.get_peft_model(model, **LORA)
docs = hl_lib.load_jsonl('cpt_train.jsonl')
blocks = hl_lib.pack_blocks((tok(d['text'], add_special_tokens=False)['input_ids'] for d in docs), BLOCK, tok.eos_token_id, budget=TOKEN_BUDGET)
train_b, val_b = blocks[:-40], blocks[-40:]           # last 40 blocks: validation loss curve (unseen text, same distribution)
print(len(train_b), 'train blocks =', len(train_b) * BLOCK / 1e6, 'M tokens;', len(val_b), 'validation blocks')
mk = lambda b: Dataset.from_dict({'input_ids': b, 'labels': b})

In [ ]:
args = TrainingArguments(output_dir=f'{BASE}/cpt_ckpt', per_device_train_batch_size=1, gradient_accumulation_steps=16, per_device_eval_batch_size=1,
    learning_rate=1e-4, lr_scheduler_type='cosine', warmup_steps=10, num_train_epochs=1, optim='adamw_8bit', weight_decay=0.01,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), logging_steps=5, eval_strategy='steps', eval_steps=25,
    save_steps=25, save_total_limit=2, report_to='none', seed=3407)
trainer = Trainer(model=model, args=args, train_dataset=mk(train_b), eval_dataset=mk(val_b), data_collator=default_data_collator)
trainer.train(resume_from_checkpoint=bool(glob.glob(f'{BASE}/cpt_ckpt/checkpoint-*')))
model.save_pretrained(f'{BASE}/adapter_cpt'); tok.save_pretrained(f'{BASE}/adapter_cpt')
json.dump(trainer.state.log_history, open(f'{BASE}/cpt_log.json', 'w'))
print('saved', f'{BASE}/adapter_cpt')

In [ ]:
import matplotlib.pyplot as plt
log = json.load(open(f'{BASE}/cpt_log.json'))
tr = [(l['step'], l['loss']) for l in log if 'loss' in l]; ev = [(l['step'], l['eval_loss']) for l in log if 'eval_loss' in l]
plt.plot(*zip(*tr), label='train loss'); plt.plot(*zip(*ev), 'o-', label='validation loss'); plt.xlabel('optimizer step'); plt.ylabel('loss (nats/token)')
plt.title('Continued pre-training on health text'); plt.legend(); plt.grid(alpha=.3); plt.savefig(f'{BASE}/cpt_loss.png', dpi=150); plt.show()